# 02 — Preprocessing & Deduplication
**SuaraLens** | Cleaning teks ringan dan deteksi near-duplicate berbasis semantic embedding.


In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from modules.preprocessing import (
    clean_text, load_embedding_model, encode_texts,
    dedup_check, evaluate_dedup
)

sns.set_theme(style='whitegrid')
DATA_PATH   = '../data/suaralens_dummy_simulasi.jsonl'
OUTPUT_PATH = '../data/output/duplicates_report.json'

df = pd.read_json(DATA_PATH, lines=True)
print(f'Dataset: {len(df):,} baris')


## 1. Fungsi clean_text — Demonstrasi

In [ ]:
test_cases = [
    'WIFI LAB MATI LAGI!!!! udah 3 hari gak bisa konek ke internet di gedung D',
    'Tolong perbaiki fasilitas parkir yg sempit...  sangat tidak nyaman  ',
    'Info: UKT semester ini bisa dibayar via https://payment.pens.ac.id sampai tgl 15',
]
for t in test_cases:
    print(f'Original : {t}')
    print(f'Cleaned  : {clean_text(t)}')
    print()


## 2. Clean Semua Teks

In [ ]:
df['teks_cleaned'] = df['teks_aduan'].apply(clean_text)
print('Cleaning selesai.')
print(f'Rata-rata panjang original: {df["teks_aduan"].str.len().mean():.0f} char')
print(f'Rata-rata panjang cleaned : {df["teks_cleaned"].str.len().mean():.0f} char')


## 3. Load Model & Encode Teks

In [ ]:
model = load_embedding_model('paraphrase-multilingual-MiniLM-L12-v2')
texts = df['teks_cleaned'].tolist()
embeddings = encode_texts(model, texts, show_progress=True)
print(f'Embedding shape: {embeddings.shape}')


## 4. Deteksi Near-Duplicate (threshold=0.90)

In [ ]:
THRESHOLD = 0.90

print(f'Mencari pasangan dengan cosine similarity >= {THRESHOLD}...')
print('(Proses ini mungkin memakan beberapa menit untuk N=5000+)')

ids   = df['id_aduan'].tolist()
pairs = dedup_check(embeddings, threshold=THRESHOLD, ids=ids)

print(f'\nTotal pasangan near-duplicate terdeteksi: {len(pairs):,}')


## 5. Validasi terhadap Duplikat yang Disisipkan Generator

In [ ]:
eval_result = evaluate_dedup(pairs, known_dup_prefix='SL-1')
print('=== Evaluasi Deteksi Duplikat ===')
for k, v in eval_result.items():
    print(f'  {k}: {v}')


## 6. Contoh Pasangan Near-Duplicate

In [ ]:
print('=== 5 Pasangan Near-Duplicate Teratas ===\n')
for i, pair in enumerate(pairs[:5], 1):
    text_a = df[df['id_aduan'] == pair['id_a']]['teks_aduan'].values[0] if pair['id_a'] in df['id_aduan'].values else '?'
    text_b = df[df['id_aduan'] == pair['id_b']]['teks_aduan'].values[0] if pair['id_b'] in df['id_aduan'].values else '?'
    print(f'--- Pasangan {i} (similarity: {pair["similarity"]:.4f}) ---')
    print(f'  A [{pair["id_a"]}]: {text_a[:120]}...')
    print(f'  B [{pair["id_b"]}]: {text_b[:120]}...')
    print()


## 7. Simpan Laporan ke JSON

In [ ]:
report = {
    'generated_at':        pd.Timestamp.now().isoformat(),
    'total_records':       len(df),
    'threshold':           THRESHOLD,
    'total_pairs_found':   len(pairs),
    'evaluation':          eval_result,
    'sample_pairs':        pairs[:10],
    'threshold_rationale': (
        'Threshold 0.90 dipilih sebagai trade-off antara presisi dan recall. '
        'Nilai lebih tinggi (0.95+) meningkatkan presisi tapi melewatkan banyak duplikat; '
        'nilai lebih rendah (0.85-) menangkap lebih banyak tapi menghasilkan false positive tinggi.'
    )
}

output_path = Path(OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)

print(f'Laporan disimpan ke: {OUTPUT_PATH}')


## Catatan: Trade-off Threshold 0.90

| Threshold | Efek |
|---|---|
| > 0.95 | Presisi tinggi, banyak duplikat lolos |
| 0.90 (dipilih) | Keseimbangan presisi–recall untuk dataset ini |
| < 0.85 | Recall tinggi, banyak false positive |

Untuk produksi, threshold optimal sebaiknya dikalibrasi menggunakan anotasi manual
pada subset data nyata.
